<a href="https://colab.research.google.com/github/oleg61/--AI--/blob/main/%D0%92%D0%BE%D1%80%D0%BE%D0%BF%D0%B0%D0%B5%D0%B2_%D0%9E%D0%A1_%D0%94%D0%97_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задание

Что нужно делать?

Датасет ml-latest.

Вспомнить подходы, которые мы разбирали.

Выбрать понравившийся подход к гибридным системам.

Написать свою.

In [ ]:
!pip install lightfm

In [ ]:
from lightfm import LightFM

In [ ]:
# Импортируем необходимые библиотеки
import numpy as np

from lightfm.evaluation import precision_at_k, auc_score
from lightfm.datasets import fetch_movielens

In [ ]:
movielens = fetch_movielens()

In [ ]:
# Выведем информацию о загруженных данных
for key, value in movielens.items():
    print(f"{key}: тип = {type(value)}, размер = {value.shape if hasattr(value, 'shape') else 'N/A'}")

In [ ]:
# Разделим данные на train и test
train = movielens['train']  # Обучающая матрица взаимодействий
test = movielens['test']    # Тестовая матрица взаимодействий

In [ ]:
# Признаки фильмов (жанры: action, comedy и т.д.)
item_features = movielens['item_features']

# Признаки пользователей
user_features = movielens.get('user_features', None)

In [ ]:
#Создание и обучение базовой модели
print("\n=== Модель 1: Только коллаборативная фильтрация (без признаков) ===")

model_cf = LightFM(loss='warp', learning_rate=0.05, random_state=42)

In [ ]:
# Обучаем модель только на матрице взаимодействий
model_cf.fit(train, epochs=30, num_threads=2)

In [ ]:
# Оцениваем качество
train_precision_cf = precision_at_k(model_cf, train, k=10).mean()
test_precision_cf = precision_at_k(model_cf, test, k=10, train_interactions=train).mean()

print(f"Precision@10 — train: {train_precision_cf:.4f}, test: {test_precision_cf:.4f}")

In [ ]:
# Создание гибридной модели (с использованием признаков фильмов)
print("\n=== Модель 2: Гибридная система (коллаборативная + контентная фильтрация) ===")

model_hybrid = LightFM(loss='warp', learning_rate=0.05, random_state=42)

In [ ]:
# Обучаем модель с использованием признаков фильмов (item_features)
model_hybrid.fit(
    interactions=train,
    item_features=item_features,      # Передаём признаки фильмов
    user_features=user_features,      # Передаём признаки пользователей (если есть)
    epochs=30,
    num_threads=2
)

In [ ]:
# Оцениваем качество гибридной модели
train_precision_hybrid = precision_at_k(model_hybrid, train, k=10).mean()
test_precision_hybrid = precision_at_k(model_hybrid, test, k=10, train_interactions=train).mean()

print(f"Precision@10 — train: {train_precision_hybrid:.4f}, test: {test_precision_hybrid:.4f}")

In [ ]:
#Функция рекомендаций с пояснениями
def sample_recommendation(model, data, user_ids):
    """
    Генерирует рекомендации для заданных пользователей.

    Параметры:
    - model: обученная модель LightFM
    - data: словарь с данными (train, item_labels и др.)
    - user_ids: список ID пользователей для рекомендаций
    """
    n_users, n_items = data['train'].shape

    train_csr = data['train'].tocsr()

    for user_id in user_ids:
        known_positives = data['item_labels'][train_csr[user_id].indices]
        scores = model.predict(user_id, np.arange(n_items))
        top_items = data['item_labels'][np.argsort(-scores)]
        print(f"\nПользователь {user_id}")
        print("  Уже понравилось:")
        for x in known_positives[:3]:
            print(f"    - {x}")
        print("  Рекомендуем:")
        for x in top_items[:3]:
            print(f"    - {x}")

In [ ]:
#Генерация рекомендаций для обеих моделей
print("\n=== Рекомендации от гибридной модели ===")
sample_recommendation(model_hybrid, movielens, [10, 25, 451])

In [ ]:
#Вывод и сравнение моделей
print("\n=== ВЫВОД ===")
print(f"Коллаборативная модель: Precision@10 на тесте = {test_precision_cf:.4f}")
print(f"Гибридная модель:       Precision@10 на тесте = {test_precision_hybrid:.4f}")

if test_precision_hybrid > test_precision_cf:
    print("✅ Гибридная модель показала лучшее качество!")
    print("   Это означает, что использование признаков фильмов (жанров) улучшило рекомендации.")
else:
    print("⚠️ Гибридная модель не улучшила результат.")
    print("   Возможно, признаки неинформативны или модель переобучена.")